In [6]:
# ======================================================================
# REVIEWER 1 / PROBLEM 3
# FINAL ONE-CELL GOOGLE COLAB CODE
#
# STRICT-FAIR MORPHOLOGICAL PREPROCESSING BASELINE COMPARISON
#
# CONDITIONS:
#   1. CLEAN
#   2. APERTIUM-KAZ token-level lemma normalization
#   3. KazNLP AnalyzerDD + TaggerHMM normalization
#   4. RELATIONAL CSE
#
# MODELS:
#   1. sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
#   2. BAAI/bge-m3
#   3. intfloat/multilingual-e5-large-instruct
#
# FIXED STRICT-FAIR PROTOCOL:
#   Total paired = 6,759
#   Train        = 6,083
#   Test         = 676
#   Seed         = 42
#
# SAME CLEAN candidate answers and gold answers for ALL conditions.
# NO FINE-TUNING.
#
# IMPORTANT FIXES:
#   - Apertium words are analyzed independently / punctuation-separated
#     so multiword units such as "бірдей ме" cannot break alignment.
#   - BERTScore uses rescale_with_baseline=False because there is no
#     official kk rescaling baseline for bert-base-multilingual-cased.
#   - E5 uses batch_size=24, exactly as in the validated Problem-2 run.
#
# METRICS:
#   Exact@1
#   TokenF1@1
#   QSim
#   Semantic@1 (answer cosine >= 0.85)
#   BERTScoreF1@1
#
# STATISTICS:
#   continuous metrics:
#       paired two-sided sign-flip permutation, 10,000
#   binary metrics:
#       exact two-sided McNemar
#   confidence interval:
#       paired bootstrap, 10,000
#   multiple testing:
#       Holm within five metrics per comparison
#       + Holm across 10 primary tests/model
#
# REQUIRED FILES:
#   baseline_15000.json
#   kazakh_segmented_15000.json
# ======================================================================


# ======================================================================
# 0. ENVIRONMENT
# ======================================================================

import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ.setdefault(
    "PYTORCH_ALLOC_CONF",
    "expandable_segments:True"
)

import sys
import re
import gc
import json
import glob
import random
import hashlib
import shutil
import zipfile
import warnings
import subprocess
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")


# ======================================================================
# 1. INSTALLATION HELPERS
# ======================================================================

def run_cmd(cmd, check=True, capture=False, cwd=None):

    result = subprocess.run(
        cmd,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
        cwd=cwd,
    )

    if check and result.returncode != 0:

        print("\nCOMMAND FAILED:")
        print(cmd)

        if capture:
            print("\nSTDOUT:")
            print((result.stdout or "")[-4000:])

            print("\nSTDERR:")
            print((result.stderr or "")[-4000:])

        raise RuntimeError(
            f"Command failed with code {result.returncode}"
        )

    return result


def ensure_pip(package, import_name=None):

    module_name = (
        import_name
        or package.split("==")[0].replace("-", "_")
    )

    try:
        __import__(module_name)

    except Exception:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package,
            ]
        )


print("=" * 92)
print("INSTALLING / CHECKING PYTHON PACKAGES")
print("=" * 92)

ensure_pip("numpy")
ensure_pip("pandas")
ensure_pip("scipy")
ensure_pip("scikit-learn", "sklearn")
ensure_pip(
    "sentence-transformers",
    "sentence_transformers"
)
ensure_pip(
    "bert-score",
    "bert_score"
)
ensure_pip("joblib")


# ======================================================================
# 2. IMPORTS
# ======================================================================

import numpy as np
import pandas as pd
import torch
import joblib

from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from bert_score import BERTScorer


try:
    from google.colab import files
    IN_COLAB = True

except Exception:
    files = None
    IN_COLAB = False


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    300
)

pd.set_option(
    "display.max_colwidth",
    150
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.9f}"
)


# ======================================================================
# 3. CONFIGURATION
# ======================================================================

BASE_FILENAME = "baseline_15000.json"

SEG_FILENAME = "kazakh_segmented_15000.json"


SEED = 42

TEST_SIZE = 0.10

SEM_THR = 0.85


N_BOOT = 10_000

N_PERM = 10_000

STAT_ALPHA = 0.05

STAT_SEED = 20260901


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


EXPECTED = {

    "base_records": 14991,

    "seg_records": 14998,

    "base_unique": 14689,

    "seg_unique": 14696,

    "paired": 6759,

    "train": 6083,

    "test": 676,
}


EXPECTED_SPLIT_SHA256 = (
    "867d3305fbc71068afedeae71b4ef102"
    "21968d23270202bf0eda860fc8bf880b"
)


E5_QUERY_INSTRUCTION = (
    "Instruct: Given a question in Kazakh, "
    "retrieve the most relevant answer.\n"
    "Query: "
)


# EXACT inference batch sizes used for validated pipeline.
MODEL_SPECS = {

    "MiniLM": {

        "name":
            "sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2",

        "batch": 64,

        "e5": False,
    },


    "BGE-M3": {

        "name":
            "BAAI/bge-m3",

        "batch": 12,

        "e5": False,
    },


    "E5-large": {

        "name":
            "intfloat/"
            "multilingual-e5-large-instruct",

        # IMPORTANT:
        # same batch as validated Problem-2 E5 run
        "batch": 32,

        "e5": True,
    },
}


OUT_DIR = Path(
    "/content/"
    "reviewer1_problem3_final"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ZIP_PATH = Path(
    "/content/"
    "reviewer1_problem3_final_outputs.zip"
)


def set_all_seeds(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


print("\nDEVICE =", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU    =",
        torch.cuda.get_device_name(0)
    )

else:

    raise RuntimeError(
        "GPU not detected. "
        "Select Runtime -> Change runtime type -> T4 GPU."
    )


# ======================================================================
# 4. FIND / UPLOAD INPUT FILES
# ======================================================================

def find_file(filename):

    candidates = [

        filename,

        f"/content/{filename}",

        f"/mnt/data/{filename}",
    ]


    for path in candidates:

        if Path(path).is_file():

            return str(
                Path(path).resolve()
            )


    matches = glob.glob(
        f"/content/**/{filename}",
        recursive=True
    )


    matches = [
        p
        for p in matches
        if Path(p).is_file()
    ]


    if matches:

        return str(
            Path(matches[0]).resolve()
        )


    return ""


base_path = find_file(
    BASE_FILENAME
)

seg_path = find_file(
    SEG_FILENAME
)


if not base_path or not seg_path:

    if IN_COLAB:

        print(
            "\nUPLOAD BOTH FILES:\n"
            "1. baseline_15000.json\n"
            "2. kazakh_segmented_15000.json\n"
        )

        files.upload()

        base_path = find_file(
            BASE_FILENAME
        )

        seg_path = find_file(
            SEG_FILENAME
        )


if not base_path or not seg_path:

    raise FileNotFoundError(
        "Required JSON files not found."
    )


# ======================================================================
# 5. ROBUST QA JSON LOADER
# ======================================================================

def normalize_records(data):

    if not isinstance(data, list):

        raise ValueError(
            "QA data must be a list."
        )


    rows = []


    for item in data:

        if not isinstance(item, dict):

            continue


        question = (
            item.get("question")
            or item.get("instruction")
            or ""
        )


        answer = (
            item.get("answer")
            or item.get("response")
            or ""
        )


        question = str(
            question
        ).strip()


        answer = str(
            answer
        ).strip()


        if question and answer:

            rows.append(
                {
                    "question": question,
                    "answer": answer,
                }
            )


    if not rows:

        raise ValueError(
            "No valid QA records found."
        )


    return rows


def load_qa(path):

    text = Path(path).read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()


    # Standard JSON array
    try:

        parsed = json.loads(text)

        if isinstance(parsed, list):

            return normalize_records(
                parsed
            )

    except Exception:

        pass


    # JSONL
    try:

        objects = []

        for line in text.splitlines():

            line = line.strip().rstrip(",")

            if line:

                objects.append(
                    json.loads(line)
                )


        if objects:

            return normalize_records(
                objects
            )

    except Exception:

        pass


    # Robust object scanner
    objects = []

    buffer = []

    depth = 0

    in_string = False

    escaped = False

    started = False


    for ch in text:

        if not started:

            if ch == "{":

                started = True

                depth = 1

                buffer = ["{"]

            continue


        buffer.append(ch)


        if in_string:

            if escaped:

                escaped = False

            elif ch == "\\":

                escaped = True

            elif ch == '"':

                in_string = False


        else:

            if ch == '"':

                in_string = True

            elif ch == "{":

                depth += 1

            elif ch == "}":

                depth -= 1


                if depth == 0:

                    raw = "".join(
                        buffer
                    )

                    buffer = []

                    started = False


                    try:

                        objects.append(
                            json.loads(raw)
                        )

                    except Exception:

                        pass


    return normalize_records(
        objects
    )


base_rows = load_qa(
    base_path
)

seg_rows = load_qa(
    seg_path
)


print("\n" + "=" * 92)
print("SOURCE DATA")
print("=" * 92)

print(
    "BASE records =",
    f"{len(base_rows):,}"
)

print(
    "SEG  records =",
    f"{len(seg_rows):,}"
)


# ======================================================================
# 6. EXACT STRICT-FAIR NORMALIZATION
# ======================================================================

_punct_left = re.compile(
    r"\s+([.,!?;:%)\]\}])"
)

_punct_right = re.compile(
    r"([(\[\{])\s+"
)

_multi_space = re.compile(
    r"\s+"
)


def norm_space_punct(text):

    text = str(text)

    text = text.replace(
        " - ",
        "-"
    )

    text = _punct_left.sub(
        r"\1",
        text
    )

    text = _punct_right.sub(
        r"\1",
        text
    )

    text = _multi_space.sub(
        " ",
        text
    ).strip()

    return text


def morph_marker_view(text):

    return norm_space_punct(
        str(text)
    )


def clean_view(text):

    text = str(text)

    text = text.replace(
        "@@ ",
        ""
    )

    text = text.replace(
        "@@",
        ""
    )

    return norm_space_punct(
        text
    )


def norm_for_exact(text):

    return re.sub(
        r"\s+",
        " ",
        norm_space_punct(
            str(text)
        ).lower()
    ).strip()


# ======================================================================
# 7. TOKEN F1
# ======================================================================

TOKEN_RE = re.compile(
    r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+"
)


def metric_tokens(text):

    return TOKEN_RE.findall(
        norm_space_punct(
            str(text)
        ).lower()
    )


def token_f1(pred, gold):

    pred_tokens = metric_tokens(
        pred
    )

    gold_tokens = metric_tokens(
        gold
    )


    if not pred_tokens and not gold_tokens:

        return 1.0


    if not pred_tokens or not gold_tokens:

        return 0.0


    pred_counter = Counter(
        pred_tokens
    )

    gold_counter = Counter(
        gold_tokens
    )


    overlap = sum(
        (
            pred_counter
            &
            gold_counter
        ).values()
    )


    if overlap == 0:

        return 0.0


    precision = (
        overlap
        /
        len(pred_tokens)
    )


    recall = (
        overlap
        /
        len(gold_tokens)
    )


    return float(
        2
        *
        precision
        *
        recall
        /
        (
            precision
            +
            recall
            +
            1e-12
        )
    )


# ======================================================================
# 8. BUILD EXACT SAME 6,759 PAIRS
# ======================================================================

def first_by_key(rows):

    mapping = {}


    for row in rows:

        key = clean_view(
            row["question"]
        )


        if key and key not in mapping:

            mapping[key] = row


    return mapping


base_map = first_by_key(
    base_rows
)

seg_map = first_by_key(
    seg_rows
)


common_keys = sorted(
    set(base_map)
    &
    set(seg_map)
)


paired = []


for pair_id, key in enumerate(
    common_keys
):

    clean_row = base_map[
        key
    ]

    segmented_row = seg_map[
        key
    ]


    paired.append(
        {
            "pair_id":
                pair_id,

            "pair_key":
                key,

            "clean_q":
                clean_view(
                    clean_row["question"]
                ),

            "cse_q":
                morph_marker_view(
                    segmented_row["question"]
                ),

            # FIXED CLEAN ANSWER SPACE
            "clean_a":
                norm_space_punct(
                    clean_row["answer"]
                ),
        }
    )


counts = {

    "base_records":
        len(base_rows),

    "seg_records":
        len(seg_rows),

    "base_unique":
        len(base_map),

    "seg_unique":
        len(seg_map),

    "paired":
        len(paired),
}


print("\n" + "=" * 92)
print("STRICT-FAIR PAIRING AUDIT")
print("=" * 92)


for key, value in counts.items():

    print(
        f"{key:<18} = "
        f"{value:,}"
    )


for key in [

    "base_records",

    "seg_records",

    "base_unique",

    "seg_unique",

    "paired",
]:

    if counts[key] != EXPECTED[key]:

        raise RuntimeError(
            f"STOP: {key}={counts[key]}, "
            f"expected {EXPECTED[key]}"
        )


print(
    "✅ Exact strict-fair pairing reproduced."
)


# ======================================================================
# 9. EXACT SAME 6083 / 676 SPLIT
# ======================================================================

train_rows, test_rows = train_test_split(
    paired,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)


if (
    len(train_rows) != EXPECTED["train"]
    or
    len(test_rows) != EXPECTED["test"]
):

    raise RuntimeError(
        "Unexpected train/test sizes."
    )


train_keys = sorted(
    row["pair_key"]
    for row in train_rows
)

test_keys = sorted(
    row["pair_key"]
    for row in test_rows
)


split_sha = hashlib.sha256(
    (
        "\n".join(train_keys)
        +
        "\n---TEST---\n"
        +
        "\n".join(test_keys)
    ).encode("utf-8")
).hexdigest()


print("\n" + "=" * 92)
print("STRICT-FAIR SPLIT")
print("=" * 92)

print(
    "Total =",
    len(paired)
)

print(
    "Train =",
    len(train_rows)
)

print(
    "Test  =",
    len(test_rows)
)

print(
    "Seed  =",
    SEED
)

print(
    "SHA256 =",
    split_sha
)


if split_sha != EXPECTED_SPLIT_SHA256:

    raise RuntimeError(
        "STOP: split SHA differs from "
        "validated Problem 1/2 split."
    )


print(
    "✅ Exact same Problem 1/2 split reproduced."
)


# ======================================================================
# 10. INSTALL APERTIUM-KAZ
# ======================================================================

print("\n" + "=" * 92)
print("SETTING UP APERTIUM-KAZ")
print("=" * 92)


def apertium_available():

    if not shutil.which(
        "apertium"
    ):

        return False


    try:

        result = run_cmd(
            ["apertium", "-l"],
            check=False,
            capture=True,
        )


        output = (
            (result.stdout or "")
            +
            (result.stderr or "")
        )


        return (
            "kaz-tagger"
            in output
        )

    except Exception:

        return False


APERTIUM_LOCAL_DIR = Path(
    "/content/apertium-kaz"
)


if not apertium_available():

    print(
        "Installing Apertium..."
    )


    run_cmd(
        "apt-get update -qq"
    )


    run_cmd(
        "apt-get install -y -qq "
        "curl git build-essential "
        "automake autoconf libtool "
        "pkg-config gawk"
    )


    run_cmd(
        "apt-get install -y -qq "
        "apertium apertium-kaz",
        check=False
    )


if not apertium_available():

    print(
        "Adding official Apertium repository..."
    )


    run_cmd(
        "curl -sS "
        "https://apertium.projectjj.com/apt/"
        "install-nightly.sh | bash"
    )


    run_cmd(
        "apt-get update -qq"
    )


    run_cmd(
        "apt-get install -y -qq "
        "apertium-all-dev apertium-kaz",
        check=False
    )


if not apertium_available():

    print(
        "Package unavailable; "
        "building apertium-kaz from source."
    )


    run_cmd(
        "apt-get install -y -qq "
        "git apertium-all-dev"
    )


    if not (
        APERTIUM_LOCAL_DIR
        /
        ".git"
    ).exists():

        if APERTIUM_LOCAL_DIR.exists():

            shutil.rmtree(
                APERTIUM_LOCAL_DIR
            )


        run_cmd(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/"
                "apertium/apertium-kaz.git",
                str(
                    APERTIUM_LOCAL_DIR
                ),
            ]
        )


    run_cmd(
        "./autogen.sh",
        cwd=str(
            APERTIUM_LOCAL_DIR
        )
    )


    run_cmd(
        "make -j2",
        cwd=str(
            APERTIUM_LOCAL_DIR
        )
    )


if apertium_available():

    APERTIUM_TAGGER_CMD = [
        "apertium",
        "kaz-tagger",
    ]

    APERTIUM_SOURCE = (
        "installed apertium-kaz package"
    )

else:

    APERTIUM_TAGGER_CMD = [
        "apertium",
        "-d",
        str(APERTIUM_LOCAL_DIR),
        "kaz-tagger",
    ]

    APERTIUM_SOURCE = (
        "local apertium-kaz source build"
    )


ap_test = subprocess.run(
    APERTIUM_TAGGER_CMD,
    input="қазақ тілі\n",
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)


if ap_test.returncode != 0:

    raise RuntimeError(
        "Apertium test failed:\n"
        +
        (ap_test.stderr or "")[-3000:]
    )


print(
    "✅ Apertium ready:"
)

print(
    "   source =",
    APERTIUM_SOURCE
)

print(
    "   sample =",
    ap_test.stdout.strip()[:500]
)


# ======================================================================
# 11. INSTALL KAZNLP
# ======================================================================

print("\n" + "=" * 92)
print("SETTING UP KAZNLP")
print("=" * 92)


run_cmd(
    "apt-get update -qq && "
    "apt-get install -y -qq git"
)


KAZNLP_REPO = Path(
    "/content/kaznlp_repo"
)


if not (
    KAZNLP_REPO
    /
    "kaznlp"
).is_dir():

    if KAZNLP_REPO.exists():

        shutil.rmtree(
            KAZNLP_REPO
        )


    run_cmd(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/"
            "nlacslab/kaznlp.git",
            str(KAZNLP_REPO),
        ]
    )


# ======================================================================
# 12. KAZNLP MODERN PYTHON COMPATIBILITY
# ======================================================================

import collections
import collections.abc


for name in [

    "Mapping",

    "MutableMapping",

    "Sequence",

    "Iterable",
]:

    if not hasattr(
        collections,
        name
    ):

        setattr(
            collections,
            name,
            getattr(
                collections.abc,
                name
            )
        )


legacy_numpy = {

    "int": int,

    "float": float,

    "bool": bool,

    "object": object,

    "str": str,
}


for name, value in legacy_numpy.items():

    if not hasattr(
        np,
        name
    ):

        try:

            setattr(
                np,
                name,
                value
            )

        except Exception:

            pass


if not hasattr(
    np,
    "unicode_"
):

    try:

        np.unicode_ = np.str_

    except Exception:

        pass


try:

    import sklearn.externals

    setattr(
        sklearn.externals,
        "joblib",
        joblib
    )

    sys.modules[
        "sklearn.externals.joblib"
    ] = joblib

except Exception:

    pass


if str(KAZNLP_REPO) not in sys.path:

    sys.path.insert(
        0,
        str(KAZNLP_REPO)
    )


from kaznlp.morphology.analyzers import AnalyzerDD
from kaznlp.morphology.taggers import TaggerHMM


kaz_analyzer = AnalyzerDD()


KAZ_MDL = (
    KAZNLP_REPO
    /
    "kaznlp"
    /
    "morphology"
    /
    "mdl"
)


kaz_analyzer.load_model(
    str(KAZ_MDL)
)


kaz_tagger = TaggerHMM(
    lyzer=kaz_analyzer
)


kaz_tagger.load_model(
    str(KAZ_MDL)
)


kaz_test_words = [
    "еңбек",
    "етсең",
    "ерінбей",
]


kaz_test = list(
    kaz_tagger.tag_sentence(
        [
            x.lower()
            for x in kaz_test_words
        ]
    )
)


if len(kaz_test) != 3:

    raise RuntimeError(
        "KazNLP test failed."
    )


print(
    "✅ KazNLP AnalyzerDD + TaggerHMM ready."
)

print(
    "   sample analyses =",
    [
        str(x)
        for x in kaz_test
    ]
)


# ======================================================================
# 13. CYRILLIC WORD HANDLING
# ======================================================================

CYR_WORD_RE = re.compile(
    r"[А-Яа-яЁё"
    r"ӘәҒғҚқҢңӨөҰұҮүҺһІі]+"
)


def get_cyr_word_matches(text):

    return list(
        CYR_WORD_RE.finditer(
            str(text)
        )
    )


def reconstruct_with_replacements(
    text,
    matches,
    replacements
):

    if len(matches) != len(replacements):

        raise ValueError(
            "Replacement count mismatch."
        )


    output = []

    last = 0


    for match, replacement in zip(
        matches,
        replacements
    ):

        output.append(
            text[
                last:
                match.start()
            ]
        )

        output.append(
            str(replacement)
        )

        last = match.end()


    output.append(
        text[last:]
    )


    return norm_space_punct(
        "".join(output)
    )


# ======================================================================
# 14. APERTIUM PARSER
# ======================================================================

def split_unescaped(
    text,
    delimiter="/"
):

    parts = []

    buffer = []

    escaped = False


    for char in text:

        if escaped:

            buffer.append(
                char
            )

            escaped = False

            continue


        if char == "\\":

            escaped = True

            buffer.append(
                char
            )

            continue


        if char == delimiter:

            parts.append(
                "".join(buffer)
            )

            buffer = []

        else:

            buffer.append(
                char
            )


    parts.append(
        "".join(buffer)
    )


    return parts


def apertium_unescape(text):

    return (
        str(text)
        .replace(r"\/", "/")
        .replace(r"\^", "^")
        .replace(r"\$", "$")
        .replace(r"\\", "\\")
    )


APERTIUM_UNIT_RE = re.compile(
    r"\^((?:\\.|[^$])*)\$"
)


def parse_apertium_output(text):

    units = APERTIUM_UNIT_RE.findall(
        str(text)
    )


    results = []


    for unit in units:

        parts = split_unescaped(
            unit,
            "/"
        )


        if len(parts) < 2:

            continue


        surface = apertium_unescape(
            parts[0]
        ).strip()


        # Keep only one-word Cyrillic lexical units.
        if not CYR_WORD_RE.fullmatch(
            surface
        ):

            continue


        analyses = parts[
            1:
        ]


        analysis = next(
            (
                x
                for x in analyses
                if x
            ),
            None
        )


        if analysis is None:

            results.append(
                (
                    surface.lower(),
                    False
                )
            )

            continue


        analysis = apertium_unescape(
            analysis
        )


        if analysis.startswith("*"):

            results.append(
                (
                    surface.lower(),
                    False
                )
            )

            continue


        lemma = (
            analysis
            .split(
                "<",
                1
            )[0]
            .strip()
            .lstrip("#")
            .lower()
        )


        if not lemma:

            lemma = surface.lower()


        results.append(
            (
                lemma,
                True
            )
        )


    return results


# ======================================================================
# 15. APERTIUM TOKEN-LEVEL LEMMATIZATION
#
# CRITICAL FIX:
# Every word is separated by sentence punctuation in the temporary
# Apertium stream. This prevents "бірдей ме" -> one lexical unit.
# ======================================================================

def apertium_analyze_unique_words(
    unique_words,
    batch_size=500
):

    unique_words = list(
        dict.fromkeys(
            [
                str(word).lower()
                for word in unique_words
                if str(word).strip()
            ]
        )
    )


    result_map = {}

    total = len(
        unique_words
    )


    print(
        "Unique Cyrillic surface forms =",
        f"{total:,}"
    )


    for start in range(
        0,
        total,
        batch_size
    ):

        batch = unique_words[
            start:
            start + batch_size
        ]


        # Each input word becomes its own mini sentence:
        # сөз . сөз . сөз .
        stream = (
            " . ".join(batch)
            +
            " .\n"
        )


        process = subprocess.run(
            APERTIUM_TAGGER_CMD,
            input=stream,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )


        if process.returncode != 0:

            raise RuntimeError(
                "Apertium batch failed:\n"
                +
                (process.stderr or "")[-3000:]
            )


        parsed = parse_apertium_output(
            process.stdout
        )


        # Normally one parsed lexical unit per input word.
        if len(parsed) != len(batch):

            print(
                "\n⚠️ Batch alignment mismatch detected."
            )

            print(
                "Expected =",
                len(batch)
            )

            print(
                "Parsed   =",
                len(parsed)
            )

            print(
                "Falling back to individual "
                "word analysis for this batch."
            )


            parsed = []


            for word in batch:

                individual = subprocess.run(
                    APERTIUM_TAGGER_CMD,
                    input=word + " .\n",
                    text=True,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                )


                if individual.returncode != 0:

                    parsed.append(
                        (
                            word,
                            False
                        )
                    )

                    continue


                result = parse_apertium_output(
                    individual.stdout
                )


                if len(result) == 1:

                    parsed.append(
                        result[0]
                    )

                else:

                    parsed.append(
                        (
                            word,
                            False
                        )
                    )


        if len(parsed) != len(batch):

            raise RuntimeError(
                "Apertium alignment could not "
                "be recovered."
            )


        for surface_input, (
            lemma,
            covered
        ) in zip(
            batch,
            parsed
        ):

            if not lemma:

                lemma = surface_input

                covered = False


            result_map[
                surface_input
            ] = (
                lemma.lower(),
                bool(covered)
            )


        end = min(
            start + batch_size,
            total
        )


        print(
            "Apertium analysis: "
            f"{end:,}/{total:,}"
        )


    return result_map


def apertium_preprocess_questions(
    questions
):

    question_matches = []

    all_words = []


    for question in questions:

        matches = get_cyr_word_matches(
            question
        )

        question_matches.append(
            matches
        )


        for match in matches:

            all_words.append(
                match.group(0).lower()
            )


    print(
        "Total Cyrillic word occurrences =",
        f"{len(all_words):,}"
    )


    lemma_map = apertium_analyze_unique_words(
        all_words
    )


    outputs = []


    total_words = 0

    covered_words = 0

    changed_words = 0

    changed_questions = 0


    for index, (
        question,
        matches
    ) in enumerate(
        zip(
            questions,
            question_matches
        ),
        start=1
    ):

        words = [
            match.group(0)
            for match in matches
        ]


        lemmas = []

        coverage_flags = []


        for word in words:

            lemma, covered = lemma_map.get(
                word.lower(),
                (
                    word.lower(),
                    False
                )
            )

            lemmas.append(
                lemma
            )

            coverage_flags.append(
                covered
            )


        if words:

            transformed = reconstruct_with_replacements(
                question,
                matches,
                lemmas
            )

        else:

            transformed = norm_space_punct(
                question
            )


        outputs.append(
            transformed
        )


        total_words += len(
            words
        )


        covered_words += sum(
            int(x)
            for x in coverage_flags
        )


        changed_words += sum(
            int(
                lemma.lower()
                !=
                word.lower()
            )
            for lemma, word
            in zip(
                lemmas,
                words
            )
        )


        if (
            norm_for_exact(transformed)
            !=
            norm_for_exact(question)
        ):

            changed_questions += 1


        if (
            index % 500 == 0
            or
            index == len(questions)
        ):

            print(
                "Apertium reconstruction: "
                f"{index:,}/"
                f"{len(questions):,}"
            )


    audit = {

        "method":
            "Apertium-Kaz token-level lemma",

        "questions":
            len(questions),

        "cyrillic_words":
            total_words,

        "covered_words":
            covered_words,

        "coverage_rate":
            (
                covered_words / total_words
                if total_words
                else 0.0
            ),

        "changed_words":
            changed_words,

        "changed_word_rate":
            (
                changed_words / total_words
                if total_words
                else 0.0
            ),

        "changed_questions":
            changed_questions,

        "changed_question_rate":
            (
                changed_questions
                /
                len(questions)
                if questions
                else 0.0
            ),
    }


    return (
        outputs,
        audit
    )


# ======================================================================
# 16. KAZNLP LEMMA EXTRACTION
# ======================================================================

def kaznlp_extract_lemma(
    analysis,
    original_word
):

    analysis = str(
        analysis
    ).strip()


    if not analysis:

        return (
            original_word.lower(),
            False
        )


    first = analysis.split()[0]


    # _R_X indicates unknown/unresolved form.
    if "_R_X" in first:

        return (
            original_word.lower(),
            False
        )


    if "_R_" in first:

        lemma = first.split(
            "_R_",
            1
        )[0].strip()


        if lemma:

            return (
                lemma.lower(),
                True
            )


    return (
        original_word.lower(),
        False
    )


# ======================================================================
# 17. KAZNLP PREPROCESSING
# ======================================================================

def kaznlp_preprocess_questions(
    questions
):

    outputs = []


    total_words = 0

    covered_words = 0

    changed_words = 0

    changed_questions = 0


    for index, question in enumerate(
        questions,
        start=1
    ):

        matches = get_cyr_word_matches(
            question
        )


        words = [
            match.group(0)
            for match in matches
        ]


        if not words:

            outputs.append(
                norm_space_punct(
                    question
                )
            )

            continue


        lower_words = [
            word.lower()
            for word in words
        ]


        analyses = list(
            kaz_tagger.tag_sentence(
                lower_words
            )
        )


        if len(analyses) != len(words):

            raise RuntimeError(
                "KazNLP token count mismatch "
                f"at question {index}."
            )


        parsed = [
            kaznlp_extract_lemma(
                analysis,
                word
            )
            for analysis, word
            in zip(
                analyses,
                words
            )
        ]


        lemmas = [
            item[0]
            for item in parsed
        ]


        coverage_flags = [
            item[1]
            for item in parsed
        ]


        transformed = reconstruct_with_replacements(
            question,
            matches,
            lemmas
        )


        outputs.append(
            transformed
        )


        total_words += len(
            words
        )


        covered_words += sum(
            int(x)
            for x in coverage_flags
        )


        changed_words += sum(
            int(
                lemma.lower()
                !=
                word.lower()
            )
            for lemma, word
            in zip(
                lemmas,
                words
            )
        )


        if (
            norm_for_exact(transformed)
            !=
            norm_for_exact(question)
        ):

            changed_questions += 1


        if (
            index % 250 == 0
            or
            index == len(questions)
        ):

            print(
                "KazNLP preprocessing: "
                f"{index:,}/"
                f"{len(questions):,}"
            )


    audit = {

        "method":
            "KazNLP AnalyzerDD + TaggerHMM lemma",

        "questions":
            len(questions),

        "cyrillic_words":
            total_words,

        "covered_words":
            covered_words,

        "coverage_rate":
            (
                covered_words / total_words
                if total_words
                else 0.0
            ),

        "changed_words":
            changed_words,

        "changed_word_rate":
            (
                changed_words / total_words
                if total_words
                else 0.0
            ),

        "changed_questions":
            changed_questions,

        "changed_question_rate":
            (
                changed_questions
                /
                len(questions)
                if questions
                else 0.0
            ),
    }


    return (
        outputs,
        audit
    )


# ======================================================================
# 18. CREATE APERTIUM AND KAZNLP VIEWS
# ======================================================================

print("\n" + "=" * 92)
print("PREPROCESSING STRICT-FAIR QUESTIONS")
print("=" * 92)


all_clean_questions = [
    row["clean_q"]
    for row in paired
]


print("\n--- APERTIUM-KAZ ---")

apertium_questions, apertium_audit = (
    apertium_preprocess_questions(
        all_clean_questions
    )
)


print("\n--- KAZNLP ---")

kaznlp_questions, kaznlp_audit = (
    kaznlp_preprocess_questions(
        all_clean_questions
    )
)


if (
    len(apertium_questions)
    !=
    len(paired)
    or
    len(kaznlp_questions)
    !=
    len(paired)
):

    raise RuntimeError(
        "Preprocessing output count mismatch."
    )


for index, row in enumerate(
    paired
):

    row["apertium_q"] = (
        apertium_questions[
            index
        ]
    )

    row["kaznlp_q"] = (
        kaznlp_questions[
            index
        ]
    )


# Re-create exact deterministic split after attaching views.
train_rows, test_rows = train_test_split(
    paired,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)


# ======================================================================
# 19. PREPROCESSING AUDIT
# ======================================================================

preprocess_audit_df = pd.DataFrame(
    [
        apertium_audit,
        kaznlp_audit,
    ]
)


print("\n" + "=" * 92)
print("PREPROCESSING COVERAGE AUDIT")
print("=" * 92)

print(
    preprocess_audit_df.to_string(
        index=False
    )
)


for audit in [
    apertium_audit,
    kaznlp_audit,
]:

    if (
        audit["cyrillic_words"] > 0
        and
        audit["coverage_rate"] < 0.05
    ):

        raise RuntimeError(
            f"{audit['method']} coverage "
            "is suspiciously low. "
            "Do not use the results."
        )


# ======================================================================
# 20. PREPROCESSING EXAMPLES
# ======================================================================

examples_df = pd.DataFrame(
    [
        {
            "pair_id":
                row["pair_id"],

            "CLEAN":
                row["clean_q"],

            "APERTIUM":
                row["apertium_q"],

            "KAZNLP":
                row["kaznlp_q"],

            "RELATIONAL_CSE":
                row["cse_q"],
        }

        for row in paired[:20]
    ]
)


print("\n" + "=" * 92)
print("PREPROCESSING EXAMPLES")
print("=" * 92)

print(
    examples_df
    .head(10)
    .to_string(
        index=False
    )
)


# ======================================================================
# 21. SAVE PREPROCESSED CORPUS
# ======================================================================

train_key_set = set(
    train_keys
)


preprocessed_df = pd.DataFrame(
    [
        {
            "pair_id":
                row["pair_id"],

            "pair_key":
                row["pair_key"],

            "clean_q":
                row["clean_q"],

            "apertium_q":
                row["apertium_q"],

            "kaznlp_q":
                row["kaznlp_q"],

            "cse_q":
                row["cse_q"],

            "clean_a":
                row["clean_a"],

            "split":
                (
                    "train"
                    if row["pair_key"]
                    in train_key_set
                    else "test"
                ),
        }

        for row in paired
    ]
)


preprocessed_df.to_csv(
    OUT_DIR /
    "strictfair_6759_four_question_views.csv",
    index=False,
    encoding="utf-8-sig",
)


preprocess_audit_df.to_csv(
    OUT_DIR /
    "preprocessing_coverage_audit.csv",
    index=False,
    encoding="utf-8-sig",
)


examples_df.to_csv(
    OUT_DIR /
    "preprocessing_examples.csv",
    index=False,
    encoding="utf-8-sig",
)


# ======================================================================
# 22. QUESTION VIEWS
# ======================================================================

VIEW_COLUMNS = {

    "CLEAN":
        "clean_q",

    "APERTIUM":
        "apertium_q",

    "KAZNLP":
        "kaznlp_q",

    "RELATIONAL_CSE":
        "cse_q",
}


# ======================================================================
# 23. ENCODING HELPERS
# ======================================================================

def e5_wrap_query(text):

    return (
        E5_QUERY_INSTRUCTION
        +
        str(text)
    )


def encode_questions(
    model,
    texts,
    batch_size,
    use_e5
):

    if use_e5:

        inputs = [
            e5_wrap_query(text)
            for text in texts
        ]

    else:

        inputs = list(
            texts
        )


    return model.encode(
        inputs,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def encode_answers(
    model,
    texts,
    batch_size
):

    # Plain CLEAN answers in every condition.
    return model.encode(
        list(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


# ======================================================================
# 24. RETRIEVAL EXPERIMENT
# ======================================================================

all_item_rows = []


for model_label, spec in MODEL_SPECS.items():

    print("\n\n" + "=" * 92)

    print(
        "MODEL:",
        model_label
    )

    print(
        spec["name"]
    )

    print(
        "Batch size =",
        spec["batch"]
    )

    print("=" * 92)


    set_all_seeds(
        SEED
    )


    gc.collect()

    torch.cuda.empty_cache()


    model = SentenceTransformer(
        spec["name"],
        device=DEVICE
    )


    batch_size = spec["batch"]

    use_e5 = spec["e5"]


    candidate_answers = [
        row["clean_a"]
        for row in train_rows
    ]


    gold_answers = [
        row["clean_a"]
        for row in test_rows
    ]


    print(
        "\nEncoding fixed CLEAN candidate answers..."
    )


    candidate_answer_emb = encode_answers(
        model,
        candidate_answers,
        batch_size
    )


    print(
        "Encoding fixed CLEAN gold answers..."
    )


    gold_answer_emb = encode_answers(
        model,
        gold_answers,
        batch_size
    )


    for view_name, column in VIEW_COLUMNS.items():

        print("\n" + "-" * 80)

        print(
            model_label,
            "/",
            view_name
        )

        print("-" * 80)


        train_questions = [
            row[column]
            for row in train_rows
        ]


        test_questions = [
            row[column]
            for row in test_rows
        ]


        print(
            "Encoding training questions..."
        )


        train_question_emb = encode_questions(
            model,
            train_questions,
            batch_size,
            use_e5
        )


        print(
            "Encoding test questions..."
        )


        test_question_emb = encode_questions(
            model,
            test_questions,
            batch_size,
            use_e5
        )


        similarities = np.matmul(
            test_question_emb,
            train_question_emb.T
        )


        top_indices = np.argmax(
            similarities,
            axis=1
        )


        qsim_values = similarities[
            np.arange(
                len(test_rows)
            ),
            top_indices
        ]


        for test_index, candidate_index in enumerate(
            top_indices
        ):

            candidate_index = int(
                candidate_index
            )


            test_row = test_rows[
                test_index
            ]


            predicted_answer = (
                candidate_answers[
                    candidate_index
                ]
            )


            gold_answer = (
                gold_answers[
                    test_index
                ]
            )


            exact = float(
                norm_for_exact(
                    predicted_answer
                )
                ==
                norm_for_exact(
                    gold_answer
                )
            )


            tf1 = token_f1(
                predicted_answer,
                gold_answer
            )


            answer_cosine = float(
                np.dot(
                    candidate_answer_emb[
                        candidate_index
                    ],
                    gold_answer_emb[
                        test_index
                    ]
                )
            )


            semantic_hit = float(
                answer_cosine
                >=
                SEM_THR
            )


            all_item_rows.append(
                {
                    "model":
                        model_label,

                    "model_name":
                        spec["name"],

                    "view":
                        view_name,

                    "pair_id":
                        test_row["pair_id"],

                    "pair_key":
                        test_row["pair_key"],

                    "test_question":
                        test_questions[
                            test_index
                        ],

                    "gold_answer":
                        gold_answer,

                    "pred_answer":
                        predicted_answer,

                    "retrieved_train_question":
                        train_questions[
                            candidate_index
                        ],

                    "retrieved_train_pair_id":
                        train_rows[
                            candidate_index
                        ][
                            "pair_id"
                        ],

                    "Exact":
                        exact,

                    "TokenF1":
                        tf1,

                    "QSim":
                        float(
                            qsim_values[
                                test_index
                            ]
                        ),

                    "AnsCos":
                        answer_cosine,

                    "SemHit":
                        semantic_hit,

                    "BERTScoreF1":
                        np.nan,
                }
            )


        del train_question_emb

        del test_question_emb

        del similarities


        gc.collect()

        torch.cuda.empty_cache()


    del candidate_answer_emb

    del gold_answer_emb

    del model


    gc.collect()

    torch.cuda.empty_cache()


# ======================================================================
# 25. ITEM-LEVEL DATA VALIDATION
# ======================================================================

item_df = pd.DataFrame(
    all_item_rows
)


expected_rows = (
    3
    *
    4
    *
    676
)


if len(item_df) != expected_rows:

    raise RuntimeError(
        f"Expected {expected_rows} rows, "
        f"got {len(item_df)}."
    )


print("\n" + "=" * 92)
print("RETRIEVAL COMPLETED")
print("=" * 92)

print(
    "Item-level rows =",
    f"{len(item_df):,}"
)


# Save immediately before BERTScore.
item_df.to_csv(
    OUT_DIR /
    "retrieval_results_before_bertscore.csv",
    index=False,
    encoding="utf-8-sig",
)


# ======================================================================
# 26. BERTSCORE
#
# FIX:
# There is no official rescaling baseline for:
# kk + bert-base-multilingual-cased
#
# Therefore:
# rescale_with_baseline=False
# ======================================================================

print("\n" + "=" * 92)
print("COMPUTING RAW BERTSCORE F1")
print("=" * 92)


gc.collect()

torch.cuda.empty_cache()


bert_scorer = BERTScorer(
    lang="kk",
    rescale_with_baseline=False,
    device=DEVICE,
)


BERT_CHUNK = 1024

BERT_BATCH = 16


bert_values = np.empty(
    len(item_df),
    dtype=float
)


for start in range(
    0,
    len(item_df),
    BERT_CHUNK
):

    end = min(
        start + BERT_CHUNK,
        len(item_df)
    )


    candidates = (
        item_df.iloc[
            start:end
        ]["pred_answer"]
        .astype(str)
        .tolist()
    )


    references = (
        item_df.iloc[
            start:end
        ]["gold_answer"]
        .astype(str)
        .tolist()
    )


    _, _, bert_f1 = bert_scorer.score(
        candidates,
        references,
        batch_size=BERT_BATCH
    )


    bert_values[
        start:end
    ] = (
        bert_f1
        .detach()
        .cpu()
        .numpy()
        .astype(float)
    )


    print(
        "BERTScore:",
        f"{end:,}/"
        f"{len(item_df):,}"
    )


item_df[
    "BERTScoreF1"
] = bert_values


del bert_scorer


gc.collect()

torch.cuda.empty_cache()


print(
    "✅ Raw BERTScore completed."
)


# ======================================================================
# 27. FOUR-WAY SUMMARY
# ======================================================================

summary_df = (
    item_df
    .groupby(
        [
            "model",
            "view",
        ],
        sort=False
    )
    .agg(
        N=(
            "pair_id",
            "size"
        ),

        Exact_at_1=(
            "Exact",
            "mean"
        ),

        TokenF1_at_1=(
            "TokenF1",
            "mean"
        ),

        QSim=(
            "QSim",
            "mean"
        ),

        Semantic_at_1=(
            "SemHit",
            "mean"
        ),

        BERTScoreF1_at_1=(
            "BERTScoreF1",
            "mean"
        ),
    )
    .reset_index()
)


print("\n\n" + "=" * 92)
print("TABLE A. FOUR-WAY STRICT-FAIR COMPARISON")
print("=" * 92)

print(
    summary_df.to_string(
        index=False
    )
)


# ======================================================================
# 28. PROBLEM-2 REPRODUCTION CHECK
#
# This MUST pass before Problem-3 statistics are interpreted.
# ======================================================================

EXPECTED_CORE = {

    ("MiniLM", "CLEAN"): {

        "Exact":
            0.002959,

        "TokenF1":
            0.322508,

        "QSim":
            0.867277,

        "Semantic":
            0.131657,

        "BERT":
            0.779523,
    },


    ("MiniLM", "RELATIONAL_CSE"): {

        "Exact":
            0.002959,

        "TokenF1":
            0.305317,

        "QSim":
            0.934970,

        "Semantic":
            0.105030,

        "BERT":
            0.776479,
    },


    ("BGE-M3", "CLEAN"): {

        "Exact":
            0.002959,

        "TokenF1":
            0.399269,

        "QSim":
            0.839241,

        "Semantic":
            0.162722,

        "BERT":
            0.810081,
    },


    ("BGE-M3", "RELATIONAL_CSE"): {

        "Exact":
            0.002959,

        "TokenF1":
            0.360263,

        "QSim":
            0.864946,

        "Semantic":
            0.118343,

        "BERT":
            0.797964,
    },


    ("E5-large", "CLEAN"): {

        "Exact":
            0.002959,

        "TokenF1":
            0.400414,

        "QSim":
            0.960814,

        "Semantic":
            0.991124,

        "BERT":
            0.810350,
    },


    ("E5-large", "RELATIONAL_CSE"): {

        "Exact":
            0.004438,

        "TokenF1":
            0.367864,

        "QSim":
            0.967879,

        "Semantic":
            0.982249,

        "BERT":
            0.800921,
    },
}


print("\n" + "=" * 92)
print("PROBLEM 2 REPRODUCTION CHECK")
print("=" * 92)


reproduction_rows = []

core_ok = True


for (
    model_label,
    view_name
), expected_values in EXPECTED_CORE.items():

    subset = item_df[
        (
            item_df["model"]
            ==
            model_label
        )
        &
        (
            item_df["view"]
            ==
            view_name
        )
    ]


    actual_values = {

        "Exact":
            float(
                subset["Exact"].mean()
            ),

        "TokenF1":
            float(
                subset["TokenF1"].mean()
            ),

        "QSim":
            float(
                subset["QSim"].mean()
            ),

        "Semantic":
            float(
                subset["SemHit"].mean()
            ),

        "BERT":
            float(
                subset["BERTScoreF1"].mean()
            ),
    }


    print(
        f"\n{model_label} / {view_name}"
    )


    for metric in [

        "Exact",

        "TokenF1",

        "QSim",

        "Semantic",

        "BERT",
    ]:

        # BERTScore may differ at tiny library-version level.
        tolerance = (
            1e-4
            if metric == "BERT"
            else 1e-5
        )


        difference = (
            actual_values[metric]
            -
            expected_values[metric]
        )


        matched = np.isclose(
            actual_values[metric],
            expected_values[metric],
            atol=tolerance,
            rtol=0,
        )


        print(
            f"  {metric:<10} "
            f"actual="
            f"{actual_values[metric]:.6f} "
            f"expected="
            f"{expected_values[metric]:.6f} "
            f"{'✅' if matched else '❌'}"
        )


        reproduction_rows.append(
            {
                "model":
                    model_label,

                "view":
                    view_name,

                "metric":
                    metric,

                "actual":
                    actual_values[metric],

                "expected":
                    expected_values[metric],

                "difference":
                    difference,

                "match":
                    matched,
            }
        )


        # Core retrieval metrics must reproduce.
        if (
            metric != "BERT"
            and
            not matched
        ):

            core_ok = False


reproduction_df = pd.DataFrame(
    reproduction_rows
)


reproduction_df.to_csv(
    OUT_DIR /
    "problem2_reproduction_check.csv",
    index=False,
    encoding="utf-8-sig",
)


if not core_ok:

    print("\n" + "=" * 92)

    print(
        "FAILED CORE VALUES:"
    )

    print("=" * 92)


    print(
        reproduction_df[
            (
                reproduction_df["metric"]
                !=
                "BERT"
            )
            &
            (
                ~reproduction_df["match"]
            )
        ].to_string(
            index=False
        )
    )


    raise RuntimeError(
        "STOP: validated Problem-2 core "
        "retrieval metrics were not reproduced. "
        "Problem-3 statistics are intentionally "
        "not calculated."
    )


print(
    "\n✅ ALL CLEAN/CSE CORE RETRIEVAL RESULTS "
    "REPRODUCE THE VALIDATED PROBLEM-2 PIPELINE."
)


# ======================================================================
# 29. STATISTICAL FUNCTIONS
# ======================================================================

def paired_arrays(
    reference,
    comparison
):

    reference = np.asarray(
        reference,
        dtype=float
    ).reshape(-1)


    comparison = np.asarray(
        comparison,
        dtype=float
    ).reshape(-1)


    if (
        reference.shape
        !=
        comparison.shape
        or
        reference.size
        ==
        0
    ):

        raise ValueError(
            "Invalid paired arrays."
        )


    if (
        not np.all(
            np.isfinite(reference)
        )
        or
        not np.all(
            np.isfinite(comparison)
        )
    ):

        raise ValueError(
            "NaN/Inf detected."
        )


    return (
        reference,
        comparison
    )


def paired_bootstrap_ci(
    reference,
    comparison,
    n_boot=N_BOOT,
    alpha=STAT_ALPHA,
    seed=STAT_SEED
):

    reference, comparison = paired_arrays(
        reference,
        comparison
    )


    difference = (
        comparison
        -
        reference
    )


    observed = float(
        np.mean(
            difference
        )
    )


    n = len(
        difference
    )


    rng = np.random.default_rng(
        seed
    )


    values = np.empty(
        n_boot,
        dtype=float
    )


    chunk = 1000

    position = 0


    while position < n_boot:

        current = min(
            chunk,
            n_boot - position
        )


        indices = rng.integers(
            0,
            n,
            size=(
                current,
                n
            )
        )


        values[
            position:
            position + current
        ] = np.mean(
            difference[
                indices
            ],
            axis=1
        )


        position += current


    low, high = np.quantile(
        values,
        [
            alpha / 2,
            1 - alpha / 2,
        ]
    )


    return (
        observed,
        float(low),
        float(high),
    )


def paired_signflip_p(
    reference,
    comparison,
    n_perm=N_PERM,
    seed=STAT_SEED
):

    reference, comparison = paired_arrays(
        reference,
        comparison
    )


    difference = (
        comparison
        -
        reference
    )


    if np.allclose(
        difference,
        0.0
    ):

        return 1.0


    observed = abs(
        float(
            np.mean(
                difference
            )
        )
    )


    rng = np.random.default_rng(
        seed
    )


    n = len(
        difference
    )


    extreme = 0

    done = 0

    chunk = 1000


    while done < n_perm:

        current = min(
            chunk,
            n_perm - done
        )


        signs = rng.choice(
            np.array(
                [
                    -1.0,
                    1.0
                ]
            ),
            size=(
                current,
                n
            )
        )


        permuted = np.abs(
            np.mean(
                signs
                *
                difference,
                axis=1
            )
        )


        extreme += int(
            np.sum(
                permuted
                >=
                observed - 1e-15
            )
        )


        done += current


    return float(
        (
            extreme + 1
        )
        /
        (
            n_perm + 1
        )
    )


def exact_mcnemar(
    reference,
    comparison
):

    reference = np.asarray(
        reference,
        dtype=int
    )


    comparison = np.asarray(
        comparison,
        dtype=int
    )


    # reference=1, comparison/CSE=0
    n10 = int(
        np.sum(
            (
                reference == 1
            )
            &
            (
                comparison == 0
            )
        )
    )


    # reference=0, comparison/CSE=1
    n01 = int(
        np.sum(
            (
                reference == 0
            )
            &
            (
                comparison == 1
            )
        )
    )


    discordant = (
        n10
        +
        n01
    )


    if discordant == 0:

        return (
            n10,
            n01,
            1.0
        )


    p_value = binomtest(
        n10,
        n=discordant,
        p=0.5,
        alternative="two-sided",
    ).pvalue


    return (
        n10,
        n01,
        float(p_value),
    )


def holm_adjust(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    adjusted_sorted = np.empty(
        m,
        dtype=float
    )


    running_max = 0.0


    for rank, index in enumerate(
        order
    ):

        candidate = (
            (
                m - rank
            )
            *
            p_values[index]
        )


        running_max = max(
            running_max,
            candidate
        )


        adjusted_sorted[
            rank
        ] = min(
            1.0,
            running_max
        )


    adjusted = np.empty(
        m,
        dtype=float
    )


    for rank, index in enumerate(
        order
    ):

        adjusted[
            index
        ] = adjusted_sorted[
            rank
        ]


    return adjusted


# ======================================================================
# 30. PRIMARY CSE VS APERTIUM / KAZNLP COMPARISONS
# ======================================================================

PRIMARY_COMPARISONS = [

    (
        "APERTIUM",
        "RELATIONAL_CSE",
        "CSE_minus_APERTIUM"
    ),


    (
        "KAZNLP",
        "RELATIONAL_CSE",
        "CSE_minus_KAZNLP"
    ),
]


METRICS = [

    (
        "Exact@1",
        "Exact",
        "binary"
    ),


    (
        "TokenF1@1",
        "TokenF1",
        "continuous"
    ),


    (
        "QSim",
        "QSim",
        "continuous"
    ),


    (
        "Semantic@1",
        "SemHit",
        "binary"
    ),


    (
        "BERTScoreF1@1",
        "BERTScoreF1",
        "continuous"
    ),
]


stat_rows = []


for model_index, model_label in enumerate(
    MODEL_SPECS.keys()
):

    model_df = item_df[
        item_df["model"]
        ==
        model_label
    ]


    for comparison_index, (
        reference_view,
        cse_view,
        comparison_name
    ) in enumerate(
        PRIMARY_COMPARISONS
    ):

        reference_df = (
            model_df[
                model_df["view"]
                ==
                reference_view
            ]
            .sort_values(
                "pair_id"
            )
            .reset_index(
                drop=True
            )
        )


        cse_df = (
            model_df[
                model_df["view"]
                ==
                cse_view
            ]
            .sort_values(
                "pair_id"
            )
            .reset_index(
                drop=True
            )
        )


        if (
            len(reference_df)
            !=
            676
            or
            len(cse_df)
            !=
            676
        ):

            raise RuntimeError(
                "Statistical comparison N != 676."
            )


        if not np.array_equal(
            reference_df[
                "pair_id"
            ].values,
            cse_df[
                "pair_id"
            ].values
        ):

            raise RuntimeError(
                "Pair alignment failure."
            )


        comparison_rows = []


        for metric_index, (
            metric_name,
            column,
            metric_type
        ) in enumerate(
            METRICS
        ):

            reference_values = (
                reference_df[
                    column
                ]
                .to_numpy(
                    dtype=float
                )
            )


            cse_values = (
                cse_df[
                    column
                ]
                .to_numpy(
                    dtype=float
                )
            )


            (
                delta,
                ci_low,
                ci_high
            ) = paired_bootstrap_ci(
                reference_values,
                cse_values,
                seed=(
                    STAT_SEED
                    +
                    model_index * 100
                    +
                    comparison_index * 10
                    +
                    metric_index
                )
            )


            n10 = np.nan

            n01 = np.nan


            if metric_type == "binary":

                (
                    n10,
                    n01,
                    p_raw
                ) = exact_mcnemar(
                    reference_values,
                    cse_values
                )


                test_name = (
                    "Exact two-sided McNemar"
                )


            else:

                p_raw = paired_signflip_p(
                    reference_values,
                    cse_values,
                    seed=(
                        STAT_SEED
                        +
                        10000
                        +
                        model_index * 100
                        +
                        comparison_index * 10
                        +
                        metric_index
                    )
                )


                test_name = (
                    "Two-sided paired "
                    "sign-flip permutation"
                )


            comparison_rows.append(
                {
                    "model":
                        model_label,

                    "comparison":
                        comparison_name,

                    "reference":
                        reference_view,

                    "metric":
                        metric_name,

                    "N":
                        676,

                    "reference_mean":
                        float(
                            np.mean(
                                reference_values
                            )
                        ),

                    "CSE_mean":
                        float(
                            np.mean(
                                cse_values
                            )
                        ),

                    "Delta_CSE_minus_reference":
                        delta,

                    "CI95_low":
                        ci_low,

                    "CI95_high":
                        ci_high,

                    "p_raw":
                        p_raw,

                    "test":
                        test_name,

                    "n10_reference1_CSE0":
                        n10,

                    "n01_reference0_CSE1":
                        n01,
                }
            )


        # Holm across 5 metrics per direct comparison
        corrected_five = holm_adjust(
            [
                row["p_raw"]
                for row
                in comparison_rows
            ]
        )


        for row, corrected_p in zip(
            comparison_rows,
            corrected_five
        ):

            row[
                "p_Holm_within5"
            ] = float(
                corrected_p
            )


        stat_rows.extend(
            comparison_rows
        )


stats_df = pd.DataFrame(
    stat_rows
)


# ======================================================================
# 31. CONSERVATIVE HOLM ACROSS BOTH MORPH BASELINES
#     2 comparisons x 5 metrics = 10 primary tests/model
# ======================================================================

stats_df[
    "p_Holm_primary10"
] = np.nan


for model_label in MODEL_SPECS.keys():

    indices = stats_df[
        stats_df["model"]
        ==
        model_label
    ].index


    corrected_ten = holm_adjust(
        stats_df.loc[
            indices,
            "p_raw"
        ].to_numpy()
    )


    stats_df.loc[
        indices,
        "p_Holm_primary10"
    ] = corrected_ten


stats_df[
    "Significant_primary10"
] = (
    stats_df[
        "p_Holm_primary10"
    ]
    <
    STAT_ALPHA
)


stats_df[
    "Direction"
] = np.where(
    stats_df[
        "Delta_CSE_minus_reference"
    ] > 0,

    "CSE higher",

    np.where(
        stats_df[
            "Delta_CSE_minus_reference"
        ] < 0,

        "CSE lower",

        "No difference"
    )
)


stats_df[
    "CI95"
] = stats_df.apply(
    lambda row:
        (
            f"[{row['CI95_low']:.6f}, "
            f"{row['CI95_high']:.6f}]"
        ),
    axis=1
)


display_stat_cols = [

    "model",

    "comparison",

    "metric",

    "N",

    "reference_mean",

    "CSE_mean",

    "Delta_CSE_minus_reference",

    "CI95",

    "p_raw",

    "p_Holm_within5",

    "p_Holm_primary10",

    "Significant_primary10",

    "Direction",
]


print("\n\n" + "=" * 92)

print(
    "TABLE B. PRIMARY CSE VS "
    "MORPHOLOGICAL BASELINE STATISTICS"
)

print(
    "Delta = Relational CSE - reference baseline"
)

print("=" * 92)


print(
    stats_df[
        display_stat_cols
    ].to_string(
        index=False
    )
)


# ======================================================================
# 32. DESCRIPTIVE DIFFERENCES VS CLEAN
# ======================================================================

clean_delta_rows = []


for model_label in MODEL_SPECS.keys():

    model_summary = summary_df[
        summary_df["model"]
        ==
        model_label
    ]


    clean_row = (
        model_summary[
            model_summary["view"]
            ==
            "CLEAN"
        ]
        .iloc[0]
    )


    for view in [

        "APERTIUM",

        "KAZNLP",

        "RELATIONAL_CSE",
    ]:

        row = (
            model_summary[
                model_summary["view"]
                ==
                view
            ]
            .iloc[0]
        )


        clean_delta_rows.append(
            {
                "model":
                    model_label,

                "view":
                    view,

                "Delta_Exact_vs_Clean":
                    (
                        row["Exact_at_1"]
                        -
                        clean_row["Exact_at_1"]
                    ),

                "Delta_TokenF1_vs_Clean":
                    (
                        row["TokenF1_at_1"]
                        -
                        clean_row["TokenF1_at_1"]
                    ),

                "Delta_QSim_vs_Clean":
                    (
                        row["QSim"]
                        -
                        clean_row["QSim"]
                    ),

                "Delta_Semantic_vs_Clean":
                    (
                        row["Semantic_at_1"]
                        -
                        clean_row["Semantic_at_1"]
                    ),

                "Delta_BERTScore_vs_Clean":
                    (
                        row["BERTScoreF1_at_1"]
                        -
                        clean_row[
                            "BERTScoreF1_at_1"
                        ]
                    ),
            }
        )


clean_delta_df = pd.DataFrame(
    clean_delta_rows
)


print("\n" + "=" * 92)
print("DESCRIPTIVE DELTAS VS CLEAN")
print("=" * 92)

print(
    clean_delta_df.to_string(
        index=False
    )
)


# ======================================================================
# 33. SAVE FINAL FILES
# ======================================================================

summary_path = (
    OUT_DIR
    /
    "problem3_four_way_summary.csv"
)


items_path = (
    OUT_DIR
    /
    "problem3_item_level_3models_4views.csv"
)


stats_path = (
    OUT_DIR
    /
    "problem3_CSE_vs_Apertium_KazNLP_statistics.csv"
)


clean_delta_path = (
    OUT_DIR
    /
    "problem3_descriptive_deltas_vs_clean.csv"
)


split_path = (
    OUT_DIR
    /
    "strictfair_split_manifest.csv"
)


environment_path = (
    OUT_DIR
    /
    "problem3_environment.json"
)


summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)


item_df.to_csv(
    items_path,
    index=False,
    encoding="utf-8-sig",
)


stats_df.to_csv(
    stats_path,
    index=False,
    encoding="utf-8-sig",
)


clean_delta_df.to_csv(
    clean_delta_path,
    index=False,
    encoding="utf-8-sig",
)


split_manifest = pd.DataFrame(
    [
        {
            "pair_id":
                row["pair_id"],

            "pair_key":
                row["pair_key"],

            "split":
                (
                    "train"
                    if row["pair_key"]
                    in train_key_set
                    else "test"
                ),
        }

        for row in paired
    ]
)


split_manifest.to_csv(
    split_path,
    index=False,
    encoding="utf-8-sig",
)


# ======================================================================
# 34. REPRODUCIBILITY METADATA
# ======================================================================

def get_git_commit(path):

    try:

        result = subprocess.run(
            [
                "git",
                "-C",
                str(path),
                "rev-parse",
                "HEAD",
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )


        if result.returncode == 0:

            return result.stdout.strip()

    except Exception:

        pass


    return None


kaznlp_commit = get_git_commit(
    KAZNLP_REPO
)


apertium_commit = (
    get_git_commit(
        APERTIUM_LOCAL_DIR
    )
    if APERTIUM_LOCAL_DIR.exists()
    else None
)


environment_info = {

    "reviewer_problem":
        "Reviewer 1 / Problem 3",

    "protocol":
        (
            "strict-fair morphological "
            "preprocessing baseline comparison"
        ),

    "base_records":
        len(base_rows),

    "segmented_records":
        len(seg_rows),

    "clean_unique":
        len(base_map),

    "segmented_unique":
        len(seg_map),

    "paired_total":
        6759,

    "train":
        6083,

    "test":
        676,

    "seed":
        SEED,

    "split_sha256":
        split_sha,

    "fine_tuning":
        False,

    "answer_space":
        (
            "fixed CLEAN candidate "
            "and gold answer space"
        ),

    "question_views": [

        "CLEAN",

        "APERTIUM",

        "KAZNLP",

        "RELATIONAL_CSE",
    ],

    "Apertium_method":
        (
            "Apertium-Kaz token-level "
            "dictionary-form normalization; "
            "Cyrillic surface words analyzed "
            "independently/punctuation-separated"
        ),

    "Apertium_source":
        APERTIUM_SOURCE,

    "Apertium_git_commit":
        apertium_commit,

    "KazNLP_method":
        (
            "AnalyzerDD + TaggerHMM; "
            "dictionary/root form extracted "
            "from selected analysis"
        ),

    "KazNLP_git_commit":
        kaznlp_commit,

    "models": {

        model_label: {

            "name":
                spec["name"],

            "inference_batch_size":
                spec["batch"],
        }

        for model_label, spec
        in MODEL_SPECS.items()
    },

    "E5_query_instruction":
        E5_QUERY_INSTRUCTION,

    "BERTScore":
        (
            "raw BERTScoreF1; lang=kk; "
            "rescale_with_baseline=False "
            "because no kk rescaling baseline "
            "exists for bert-base-multilingual-cased "
            "in the installed bert-score package"
        ),

    "statistics": {

        "paired_bootstrap":
            N_BOOT,

        "paired_signflip_permutation":
            N_PERM,

        "alpha":
            STAT_ALPHA,

        "statistical_seed":
            STAT_SEED,

        "Holm_within_comparison":
            "5 metrics",

        "Holm_primary_family":
            (
                "10 tests/model = "
                "2 morphology baselines x 5 metrics"
            ),
    },
}


environment_path.write_text(
    json.dumps(
        environment_info,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8"
)


# ======================================================================
# 35. ZIP EVERYTHING
# ======================================================================

if ZIP_PATH.exists():

    ZIP_PATH.unlink()


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:

    for path in OUT_DIR.iterdir():

        if path.is_file():

            archive.write(
                path,
                arcname=path.name
            )


# ======================================================================
# 36. FINAL OUTPUT
# ======================================================================

print("\n\n" + "=" * 92)
print("COPY THIS BLOCK BACK TO CHATGPT")
print("=" * 92)


print(
    "REVIEWER 1 / PROBLEM 3"
)


print(
    "Protocol = strict-fair morphological "
    "preprocessing baseline comparison"
)


print(
    "Views = CLEAN / APERTIUM / "
    "KAZNLP / RELATIONAL_CSE"
)


print(
    "Fine-tuning = NO"
)


print(
    f"Clean records = "
    f"{len(base_rows)}"
)


print(
    f"Segmented records = "
    f"{len(seg_rows)}"
)


print(
    f"Clean unique = "
    f"{len(base_map)}"
)


print(
    f"Segmented unique = "
    f"{len(seg_map)}"
)


print(
    f"Paired total = "
    f"{len(paired)}"
)


print(
    f"Train = "
    f"{len(train_rows)}"
)


print(
    f"Test = "
    f"{len(test_rows)}"
)


print(
    f"Seed = {SEED}"
)


print(
    f"Split SHA256 = {split_sha}"
)


print(
    "Answer space = fixed CLEAN "
    "for every condition"
)


print(
    "E5 batch size = 24"
)


print(
    "E5 query format = instruction-aware"
)


print(
    "BERTScore = raw F1, lang=kk, "
    "rescale_with_baseline=False"
)


print(
    "\nPREPROCESSING AUDIT:"
)


print(
    preprocess_audit_df.to_string(
        index=False
    )
)


print(
    "\nFOUR-WAY SUMMARY:"
)


print(
    summary_df.to_string(
        index=False
    )
)


print(
    "\nPROBLEM-2 REPRODUCTION CHECK:"
)


print(
    reproduction_df.to_string(
        index=False
    )
)


print(
    "\nPRIMARY CSE VS MORPHOLOGY "
    "BASELINE STATISTICS:"
)


print(
    stats_df[
        display_stat_cols
    ].to_string(
        index=False
    )
)


print(
    "\nDESCRIPTIVE DELTAS VS CLEAN:"
)


print(
    clean_delta_df.to_string(
        index=False
    )
)


print(
    "\nFILES SAVED:"
)


for path in [

    summary_path,

    items_path,

    stats_path,

    clean_delta_path,

    OUT_DIR /
    "preprocessing_coverage_audit.csv",

    OUT_DIR /
    "preprocessing_examples.csv",

    OUT_DIR /
    "strictfair_6759_four_question_views.csv",

    OUT_DIR /
    "problem2_reproduction_check.csv",

    environment_path,

    ZIP_PATH,
]:

    print(path)


print(
    "\n✅ REVIEWER 1 / PROBLEM 3 "
    "EXPERIMENT COMPLETED SUCCESSFULLY"
)

INSTALLING / CHECKING PYTHON PACKAGES

DEVICE = cuda
GPU    = Tesla T4

SOURCE DATA
BASE records = 14,991
SEG  records = 14,998

STRICT-FAIR PAIRING AUDIT
base_records       = 14,991
seg_records        = 14,998
base_unique        = 14,689
seg_unique         = 14,696
paired             = 6,759
✅ Exact strict-fair pairing reproduced.

STRICT-FAIR SPLIT
Total = 6759
Train = 6083
Test  = 676
Seed  = 42
SHA256 = 867d3305fbc71068afedeae71b4ef10221968d23270202bf0eda860fc8bf880b
✅ Exact same Problem 1/2 split reproduced.

SETTING UP APERTIUM-KAZ
✅ Apertium ready:
   source = installed apertium-kaz package
   sample = ^қазақ/қазақ<n><nom>$ ^тілі/тіл<n><px3sp><nom>+е<cop><aor><p3><sg>$^./.<sent>$

SETTING UP KAZNLP
✅ KazNLP AnalyzerDD + TaggerHMM ready.
   sample analyses = ['еңбек_R_ZE', 'ет_R_ET се_M4 ң_P2', 'ерінбей_R_X']

PREPROCESSING STRICT-FAIR QUESTIONS

--- APERTIUM-KAZ ---
Total Cyrillic word occurrences = 47,331
Unique Cyrillic surface forms = 4,465
Apertium analysis: 500/4,465
Aperti

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Encoding fixed CLEAN candidate answers...


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Encoding fixed CLEAN gold answers...


...

Batches:   0%|          | 0/22 [00:00<?, ?it/s]


RETRIEVAL COMPLETED
Item-level rows = 8,112

COMPUTING RAW BERTSCORE F1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore: 1,024/8,112
BERTScore: 2,048/8,112
BERTScore: 3,072/8,112
BERTScore: 4,096/8,112
BERTScore: 5,120/8,112
BERTScore: 6,144/8,112
BERTScore: 7,168/8,112
BERTScore: 8,112/8,112
✅ Raw BERTScore completed.


TABLE A. FOUR-WAY STRICT-FAIR COMPARISON
   model           view   N  Exact_at_1  TokenF1_at_1        QSim  Semantic_at_1  BERTScoreF1_at_1
  MiniLM          CLEAN 676 0.002958580   0.322507977 0.867277213    0.131656805       0.779522636
  MiniLM       APERTIUM 676 0.001479290   0.320392890 0.856449777    0.137573964       0.780882005
  MiniLM         KAZNLP 676 0.002958580   0.314178629 0.859711571    0.128698225       0.777503350
  MiniLM RELATIONAL_CSE 676 0.002958580   0.305317421 0.934969575    0.105029586       0.776479068
  BGE-M3          CLEAN 676 0.002958580   0.399268868 0.839241497    0.162721893       0.810081126
  BGE-M3       APERTIUM 676 0.001479290   0.362102106 0.828165352    0.125739645       0.798436057
  BGE-M3         KAZNLP 676 0.002958580   0.379815997